In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
from typing import Literal

import matplotlib.pyplot as plt

seed = 123

# Data

In [2]:
# display number given a single flatten matrix


def show(matrix: torch.Tensor):
    """Display a number given a single flatten matrix

    Args:
        matrix (torch.Tensor): 1-D array of 784 elements representing a 28x28 image

    Returns:
        plt.imshow object: Display the image
    """
    return plt.imshow(matrix.reshape(28, 28), cmap="gray")

In [3]:
train_path = "../data/train.csv"
test_path = "../data/test.csv"
submission_path = "../data/sample_submission.csv"

In [4]:
class Data:
    """attributes:
    - Train (pd.DataFrame): original train data
    - Test (pd.DataFrame): original test data
    - Submission (pd.DataFrame): original submission data
    - train (pd.DataFrame): train data after splitting
    - cv1 (pd.DataFrame): cv1 data after splitting
    - cv2 (pd.DataFrame): cv2 data after splitting
    - *x_train (torch.Tensor): x data for train
    - *y_train (torch.Tensor): y data for train
    - *x_cv1 (torch.Tensor): x data for cv1
    - *y_cv1 (torch.Tensor): y data for cv1
    - *x_cv2 (torch.Tensor): x data for cv2
    - *y_cv2 (torch.Tensor): y data for cv2
    - *test (torch.Tensor): x data for test
    - nout (int): number of unique outputs
    """

    def __init__(
        self,
        train: str,
        test: str,
        submission: str,
        cv1_size: float = 0.1,
        cv2_size: float = 0.1,
    ) -> None:
        """Data class to load the data and split it into train, test and submission dataframes

        Args:
            train (str): path for train.csv
            test (str): path for test.csv
            submission (str): path for sample_submission.csv
            cv1_size (float, optional): size of cv1 relative to train. Defaults to 0.1.
            cv2_size (float, optional): size of cv2 relative to train. Defaults to 0.1.
        """
        # import original data
        self.Train = pd.read_csv(train)
        self.Test = pd.read_csv(test)
        self.Submission = pd.read_csv(submission)

        # split data into train, cv1, cv2
        self.train, _test = train_test_split(
            self.Train, test_size=cv1_size + cv2_size, random_state=seed, stratify=self.Train["label"]
        )
        self.cv1, self.cv2 = train_test_split(
            _test, test_size=cv2_size / (cv1_size + cv2_size), random_state=seed, stratify=_test["label"]
        )
        del _test

        # split data into X, y
        self.x_train = torch.tensor(self.train.drop(columns=["label"]).values, dtype=torch.float32)
        self.y_train = torch.tensor(self.train["label"].values)
        self.x_cv1 = torch.tensor(self.cv1.drop(columns=["label"]).values, dtype=torch.float32)
        self.y_cv1 = torch.tensor(self.cv1["label"].values)
        self.x_cv2 = torch.tensor(self.cv2.drop(columns=["label"]).values, dtype=torch.float32)
        self.y_cv2 = torch.tensor(self.cv2["label"].values)
        self.test = torch.tensor(self.Test.values, dtype=torch.float32)

        # unique output
        self.nout = len(self.Train["label"].unique())

In [5]:
d = Data(
    train_path,
    test_path,
    submission_path,
    cv1_size=0.1,
    cv2_size=0.1,
)

> input shape (B,784)

# Model

In [22]:
class Model(nn.Module):
    def __init__(self, data: Data, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)

        # data class
        self.d = data

        # model
        self.sequential = nn.Sequential(
            nn.Linear(784, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 10),
        )

    def forward(self, x: torch.Tensor):
        return self.sequential(x)

    def loss(self, y_pred: torch.Tensor, y_true: torch.Tensor):
        return F.cross_entropy(y_pred, y_true)

    @torch.no_grad()
    def loss_test(self):
        return self.loss(self.forward(d.x_cv2), d.y_cv2).item()

    def summary(self) -> None:
        """print str of number of parameter to be trauned"""
        num_params = 0
        for param in self.parameters():
            num_params += param.numel()
        print(f"Number of parameters: {num_params}")

    @torch.no_grad()
    def accuracy(self, x: torch.Tensor, y: torch.Tensor) -> float:
        """calculate accuracy of model given x and y data

        Args:
            x (torch.Tensor): x data
            y (torch.Tensor): y data

        Returns:
            float: accuracy
        """
        return (self.forward(x).argmax(1) == y).float().mean().item()

    @torch.no_grad()
    def accuracy_test(self) -> float:
        """calculate accuracy of model given cv2 data

        Returns:
            float: accuracy
        """
        return self.accuracy(self.d.x_cv2, self.d.y_cv2)

    def fit(
        self,
        mode: Literal["train", "cv1", "cv2"],
        epochs: int,
        iteration: int,
        lr: float,
        print_interval: int = 1000,
    ) -> None:
        # training mode
        self.train()

        # specify data to be used
        if mode == "train":
            X = d.x_train
            Y = d.y_train
        elif mode == "cv1":
            X = d.x_cv1
            Y = d.y_cv1
        elif mode == "cv2":
            X = d.x_cv2
            Y = d.y_cv2

        # total number of element in data
        n = len(X)

        # optimizer
        optimizer = optim.Adam(self.parameters(), lr=lr)

        for epoch in range(iteration):
            optimizer.zero_grad()

            # batch
            batch_index = torch.randint(0, n, (epochs,))
            batch_x = X[batch_index]
            batch_y = Y[batch_index]

            # loss,backward,optim step
            l = self.loss(self.forward(batch_x), batch_y)
            l.backward()
            optimizer.step()

            # print every print_interval
            if ((epoch + 1) % print_interval == 0) or epoch == 0:
                if epoch == 0 :
                    self.summary()
                with torch.no_grad():
                    print(f"\n{epoch+1} Iteration")
                    print(f"Loss_Train          : {self.loss(self.forward(X), Y).item():.4f}")
                    print(f"Loss_cv2            : {self.loss_test():.4f}")
                    print(f"Accuracy_Train  : {self.accuracy(X, Y):.4f}")
                    print(f"Accuracy_cv2    : {self.accuracy_test():.4f}")

In [23]:
m = Model(data=d)

m.fit(
    mode="cv1",
    epochs=100,
    iteration=10000,
    lr=0.001,
    print_interval=5000,
)

Number of parameters: 243658

1 Iteration
Loss_Train          : 2.2099
Loss_cv2            : 2.2176
Accuracy_Train  : 0.2174
Accuracy_cv2    : 0.2188

5000 Iteration
Loss_Train          : 0.0056
Loss_cv2            : 0.3234
Accuracy_Train  : 0.9983
Accuracy_cv2    : 0.9421
